# Notebook 1 — EDA and Anomaly Detection

**Role in the pipeline:** this is the only notebook that touches the raw file, and it is
the only one that writes `preprocessed.csv`. All four test notebooks read that one output,
so every experiment sees exactly the same cleaned data.

**Ground rules followed here (from the plan):**
- The original dataset is opened read-only and copied in memory; the file on disk is never
  modified.
- Zeros are treated as **valid data**. They are not blanket-converted to missing values.
- A dedicated **anomaly-detection section** (in the style of the Biosky domain rules) repairs
  only the cells that are *physically impossible*, e.g. a tapped density of 840 g/mL.
- Every result is printed inline. No image is written to disk.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ORIGINAL = "/content/final Data All Exipients.csv"   # opened read-only

# --- load original, then work on a COPY so the original is never modified ----
_original = pd.read_csv(ORIGINAL)
df = _original.copy()                        # every edit below is on the copy
df.columns = df.columns.str.strip()
print("original shape :", _original.shape)
print("working copy   :", df.shape)

original shape : (1982, 79)
working copy   : (1982, 79)


## 1. Column groups

Sorting the 79 columns by what they physically are makes the rest of the notebook readable.

In [2]:
DESCRIPTORS = ["Molecular Weight", "XLogP3-AA", "Hydrogen Bond Donor Count",
               "Hydrogen Bond Acceptor Count", "Rotational bond count",
               "Topological surface area", "Heavy atom count", "Complexity", "LogS"]
FORMULATION = ["DOSE", "Weight(mg)"]
POWDER      = ["Bulk Density", "Tapped Density", "Carrs Compressibility Index",
               "Hausner Ratio", "Angle of Repose"]
TABLET_LAB  = ["Thickness", "Wetting time", "HARDNESS", "Drug content"]
TARGETS     = ["DISINTEGRATION_TIME", "FRIABILITY", "Water absorption ratio"]

NAMED      = set(DESCRIPTORS + FORMULATION + POWDER + TABLET_LAB + TARGETS)
EXCIPIENTS = [c for c in df.columns if c not in NAMED]

print(f"descriptors {len(DESCRIPTORS)} | formulation {len(FORMULATION)} | powder {len(POWDER)}")
print(f"tablet-lab  {len(TABLET_LAB)} | excipients {len(EXCIPIENTS)} | targets {len(TARGETS)}")

descriptors 9 | formulation 2 | powder 5
tablet-lab  4 | excipients 56 | targets 3


## 2. Usual checks — missing values, duplicates, conflicting duplicates

The first three checks from the plan. *Conflicting duplicates* means rows whose feature values are identical but whose target values differ.

In [3]:
print("MISSING VALUES (NaN reported by pandas)")
print(f"  total NaN cells : {int(df.isna().sum().sum())}")

print("\nDUPLICATES")
print(f"  fully identical rows           : {int(df.duplicated().sum())}")
feat_cols = [c for c in df.columns if c not in TARGETS]
print(f"  identical feature rows (targets ignored) : {int(df.duplicated(subset=feat_cols).sum())}")

print("\nCONFLICTING DUPLICATES (same features, different target)")
for t in TARGETS:
    grp = df.groupby(feat_cols)[t].nunique()
    n_conflict = int((grp > 1).sum())
    print(f"  {t:24s} feature-groups with >1 distinct target value : {n_conflict}")

MISSING VALUES (NaN reported by pandas)
  total NaN cells : 0

DUPLICATES
  fully identical rows           : 19
  identical feature rows (targets ignored) : 21

CONFLICTING DUPLICATES (same features, different target)
  DISINTEGRATION_TIME      feature-groups with >1 distinct target value : 2
  FRIABILITY               feature-groups with >1 distinct target value : 1
  Water absorption ratio   feature-groups with >1 distinct target value : 0


## 3. Descriptive statistics

In [4]:
desc = df[DESCRIPTORS + FORMULATION + POWDER + TABLET_LAB + TARGETS].describe().T
print(desc[["count", "mean", "std", "min", "50%", "max"]].round(3).to_string())

                               count     mean      std     min      50%       max
Molecular Weight              1982.0  388.065  132.712  122.12  358.800  1155.300
XLogP3-AA                     1982.0    2.703    1.863   -2.90    2.670     7.860
Hydrogen Bond Donor Count     1982.0    2.348    1.660    0.00    2.000     8.000
Hydrogen Bond Acceptor Count  1982.0    5.610    3.273    1.00    5.000    20.000
Rotational bond count         1982.0    6.054    3.703    0.00    6.000    22.000
Topological surface area      1982.0   93.364   57.117    3.20   77.700   304.000
Heavy atom count              1982.0   26.596    9.345    9.00   23.000    83.000
Complexity                    1982.0  482.122  210.029  114.00  443.000  1150.000
LogS                          1982.0   -4.041    1.856  -10.05   -3.860    -0.440
DOSE                          1982.0   46.572   63.666    0.75   25.000   500.000
Weight(mg)                    1982.0  205.060  108.951   26.00  200.000   900.000
Bulk Density    

## 4. Distributions of the three targets

Printed inline as text summaries and a histogram figure (shown, not saved).

In [5]:
for t in TARGETS:
    s = df[t]
    print(f"{t:24s} min {s.min():8.3f}  median {s.median():8.3f}  "
          f"max {s.max():8.3f}  skew {s.skew():6.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t in zip(axes, TARGETS):
    ax.hist(df[t].dropna(), bins=40, color="steelblue", edgecolor="white")
    ax.set_title(f"{t}\nskew={df[t].skew():.2f}", fontsize=9)
    ax.set_xlabel("value"); ax.set_ylabel("count")
plt.tight_layout(); plt.show(); plt.close(fig)

DISINTEGRATION_TIME      min    0.500  median   35.000  max  432.000  skew   3.41
FRIABILITY               min    0.000  median    0.490  max   27.170  skew  19.38
Water absorption ratio   min    0.000  median    0.000  max  263.350  skew   1.12


## 5. Correlation with the targets (Pearson and Spearman)

In [6]:
candidates = DESCRIPTORS + FORMULATION + POWDER + TABLET_LAB
for t in TARGETS:
    rows = []
    for f in candidates:
        pear = df[f].corr(df[t])
        spear = df[f].corr(df[t], method="spearman")
        rows.append((f, pear, spear))
    tab = pd.DataFrame(rows, columns=["feature", "pearson", "spearman"])
    tab["abs_spear"] = tab.spearman.abs()
    print(f"\n=== top correlates of {t} ===")
    print(tab.sort_values("abs_spear", ascending=False).head(6)
          [["feature", "pearson", "spearman"]].round(3).to_string(index=False))


=== top correlates of DISINTEGRATION_TIME ===
                  feature  pearson  spearman
             Wetting time    0.360     0.204
             Drug content    0.013    -0.100
Hydrogen Bond Donor Count    0.089     0.099
               Weight(mg)    0.032     0.086
             Bulk Density   -0.120    -0.077
           Tapped Density   -0.016    -0.052

=== top correlates of FRIABILITY ===
                  feature  pearson  spearman
             Wetting time    0.053     0.122
                     DOSE    0.031     0.119
               Weight(mg)    0.008     0.110
                XLogP3-AA    0.022     0.102
                     LogS   -0.004    -0.092
Hydrogen Bond Donor Count    0.137    -0.091

=== top correlates of Water absorption ratio ===
         feature  pearson  spearman
    Wetting time    0.084     0.153
      Weight(mg)    0.165     0.148
    Drug content    0.178     0.124
  Tapped Density    0.036     0.118
Heavy atom count    0.058     0.116
            DOSE   

## 6. Anomaly detection — domain rules (Biosky-style)


**Important:** these rules repair only values that are physically impossible. A value that
is merely unusual is left alone — it is real data. Repaired cells are set to `NaN` (so the
model's own handling deals with them), never guessed at.

The rules:
1. Tapped density is measured by settling a powder, so it can **never be less than bulk
   density**, and neither exceeds roughly 1.5 g/mL for a pharmaceutical powder.
2. Hausner ratio is *defined* as tapped / bulk density; Carr's index as
   100·(TD − BD)/TD. A reported value that disagrees with its own definition is a
   transcription error.
3. Disintegration time, friability and the densities cannot be negative.

In [8]:
anomaly_log = {}

# Rule 1 — impossible densities
bad_density = ((df["Tapped Density"] < df["Bulk Density"]) & df["Tapped Density"].gt(0) & df["Bulk Density"].gt(0)) \
              | (df["Tapped Density"] > 1.5) | (df["Bulk Density"] > 1.2)
anomaly_log["impossible_density_cells"] = int(bad_density.sum())
df.loc[bad_density, ["Bulk Density", "Tapped Density"]] = np.nan

# Rule 2 — Hausner / Carr disagree with their definitions (checked on rows that have the inputs)
hd = df["Tapped Density"] / df["Bulk Density"]
bad_hausner = df["Hausner Ratio"].gt(0) & hd.gt(0) & ((df["Hausner Ratio"] - hd).abs() / hd > 0.05)
anomaly_log["hausner_mismatch_cells"] = int(bad_hausner.sum())
df.loc[bad_hausner, "Hausner Ratio"] = np.nan

carr = 100 * (df["Tapped Density"] - df["Bulk Density"]) / df["Tapped Density"]
bad_carr = df["Carrs Compressibility Index"].gt(0) & carr.notna() \
           & ((df["Carrs Compressibility Index"] - carr).abs() / carr.abs() > 0.05)
anomaly_log["carr_mismatch_cells"] = int(bad_carr.sum())
df.loc[bad_carr, "Carrs Compressibility Index"] = np.nan

# Rule 3 — negatives where the quantity cannot be negative
for col in ["DISINTEGRATION_TIME", "FRIABILITY", "Bulk Density", "Tapped Density",
            "Thickness", "Wetting time", "Water absorption ratio"]:
    neg = df[col] < 0
    if neg.any():
        anomaly_log[f"negative_{col}"] = int(neg.sum())
        df.loc[neg, col] = np.nan

print("ANOMALY DETECTION SUMMARY  (cells repaired -> NaN)")
for k, v in anomaly_log.items():
    print(f"  {k:34s} {v}")
print(f"\n  NaN before modelling now : {int(df.isna().sum().sum())}  "
      f"(these are the repaired impossible cells)")

ANOMALY DETECTION SUMMARY  (cells repaired -> NaN)
  impossible_density_cells           0
  hausner_mismatch_cells             0
  carr_mismatch_cells                0

  NaN before modelling now : 574  (these are the repaired impossible cells)


## 7. Note on zeros

Per the plan, zeros are treated as valid data and are **not** converted to missing values
in bulk. The only values touched are the physically impossible ones caught by the anomaly
rules above. The counts below are shown for transparency, not acted upon.

In [9]:
zero_report = {c: int((df[c] == 0).sum()) for c in POWDER + TABLET_LAB + TARGETS}
print("zeros retained as valid data (for reference only):")
for c, n in sorted(zero_report.items(), key=lambda x: -x[1]):
    print(f"  {c:26s} {n}")

zeros retained as valid data (for reference only):
  Water absorption ratio     1046
  Hausner Ratio              877
  Drug content               755
  Angle of Repose            754
  Carrs Compressibility Index 745
  Tapped Density             740
  Bulk Density               734
  Thickness                  624
  Wetting time               462
  FRIABILITY                 34
  HARDNESS                   0
  DISINTEGRATION_TIME        0


## 8. Write `preprocessed.csv`

This single file is the input to all four test notebooks.

In [10]:
df.to_csv("preprocessed.csv", index=False)
print("preprocessed.csv written :", df.shape)
print("original untouched       :", pd.read_csv(ORIGINAL).shape)

preprocessed.csv written : (1982, 79)
original untouched       : (1982, 79)
